# Analyzing 10-K Filings with an LLM API

In the previous page, we used `edgartools` to download and clean three sections of Apple's fiscal 2025 10-K - _Business_, _Risk Factors_, and _Management's Discussion and Analysis_ - and saved them to `extracted/`.

In this notebook we will feed those sections into a Large Language Model through an OpenAI-compatible API and run several analytical tasks:

| #   | Task                                                         | Uses JSON schema? |
| --- | ------------------------------------------------------------ | ----------------- |
| 1   | One-paragraph executive summary                              | ❌ free-form      |
| 2   | Plain-English explanation of MD&A                            | ❌ free-form      |
| 3   | Sentiment narrative on MD&A                                  | ❌ free-form      |
| 4   | Risk-factor classification                                   | ✅ JSON schema    |
| 5   | Named-entity extraction (products, competitors, geographies) | ✅ JSON schema    |
| 6   | Final structured "executive briefing"                        | ✅ JSON schema    |

Tasks 4–6 use a structured response schema so the model returns clean JSON we can parse and use downstream - no string scraping required.

We deliberately use a small model (`gpt-5.4-mini` by default, falling back to `gpt-5.4-nano` for the cheapest tasks). You can swap in any OpenAI-compatible model - see the Switching providers section below.

:::{important} Before you run this notebook

- **Run `read-edgar-filings.ipynb` first.** This notebook reads the text files and the income statement that the first notebook writes to `extracted/`. That folder is not committed to the repository (it is listed in `.gitignore`), so it only exists after you run the first notebook yourself.
- **You need an API key, and the calls cost money.** Set `OPENAI_API_KEY` (and, for providers other than OpenAI, `OPENAI_BASE_URL`) as environment variables before starting Jupyter. Section 2 shows how. A full run makes six small requests; section 12 estimates what they cost.

:::

:::{note} About the outputs on this page

The model outputs shown here come from an earlier run with the instructor's API key. A few cells were changed or added after that run (the client setup, the cost estimate, and the verification checks in section 11), so they show no output. Run the notebook with your own key to see them.

:::


## 1. Setup and import

This notebook uses the `openai` Python package, which is pinned in `requirements.txt` at the repository root. Install the course environment once from the repository root:

```bash
python -m pip install -r requirements.txt
```

If you install `openai` on its own, put the version range in quotes so your shell does not treat `>` as a redirect:

```bash
pip install "openai>=1.40.0"
```


In [ ]:
import json
import os
import re
from pathlib import Path
from textwrap import shorten

from openai import OpenAI

## 2. Configure the OpenAI-compatible client

The `openai` Python SDK accepts two arguments that determine _which provider_ you are talking to:

- `api_key`: your secret key for that provider.
- `base_url`: the HTTPS endpoint the SDK should send requests to.

We read both from environment variables, so the key never appears in the notebook. Set them in the terminal before you start Jupyter. `OPENAI_BASE_URL` is optional: leave it unset to use OpenAI itself.

```bash
# macOS / Linux
export OPENAI_API_KEY="sk-..."
export OPENAI_BASE_URL="https://openrouter.ai/api/v1"   # only if you use OpenRouter
```

```powershell
# Windows PowerShell
$env:OPENAI_API_KEY = "sk-..."
$env:OPENAI_BASE_URL = "https://openrouter.ai/api/v1"   # only if you use OpenRouter
```

:::{warning} Never share your API keys publicly

Never share your API keys publicly, and never paste them into a notebook cell or commit them to Git. Keep them in environment variables.

You shouldn't even provide these keys to your classmates or the instructional team since they can be used to make requests that you will be billed for. If you want to share your work, clearly indicate that API calls will not work without valid keys.

:::

The client below also handles **rate limits**. Providers cap how many requests you can send per minute. When you go over, the API answers with an HTTP 429 error instead of a result. Setting `max_retries` tells the SDK to wait and try again, waiting longer after each failed attempt (exponential backoff). The same applies to timeouts and temporary server errors.


In [ ]:
api_key = os.environ.get("OPENAI_API_KEY")
base_url = os.environ.get("OPENAI_BASE_URL")  # optional: None means OpenAI's own endpoint

if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Set it as an environment variable in the terminal "
        "(see above) and start Jupyter again from there."
    )

# max_retries: when the provider answers "rate limit exceeded" (HTTP 429), times out,
# or has a temporary server error, the SDK waits and retries the request, doubling
# the wait each time. The default is 2 retries; we allow 5.
client = OpenAI(api_key=api_key, base_url=base_url, max_retries=5, timeout=120)

print("Talking to:", base_url or "https://api.openai.com/v1 (OpenAI default)")

### Choose a model

For routine financial-text tasks like summarization and extraction, a small, cheap model is plenty. We default to `gpt-5.4-mini` and use `gpt-5.4-nano` for the very cheapest task.

To use other models, set the `LLM_MODEL` and `LLM_MODEL_CHEAP` environment variables before starting Jupyter. If you are on OpenRouter, use a model identifier that exists on OpenRouter (for example, `openai/gpt-5.4-mini`). Note that free models on OpenRouter may have very limited capabilities, so you may want to use a paid model for best results. See OpenRouter's documentation for details on which models are available and their capabilities.


In [4]:
DEFAULT_MODEL = os.environ.get("LLM_MODEL", "gpt-5.4-mini")
CHEAP_MODEL = os.environ.get("LLM_MODEL_CHEAP", "gpt-5.4-nano")

print("Default model:", DEFAULT_MODEL)
print("Cheap model  :", CHEAP_MODEL)

Default model: gpt-5.4-mini
Cheap model  : gpt-5.4-nano


## 3. Load the extracted 10-K sections

We expect the previous notebook to have written three text files (plus an income statement CSV, used in section 11) into `extracted/`. If you skipped that notebook, run it first.


In [5]:
TICKER = "AAPL"
extracted_dir = Path("extracted")


def load_section(name: str) -> str:
    path = extracted_dir / f"{TICKER}_{name}.txt"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run `read-edgar-filings.ipynb` first to populate `extracted/`."
        )
    return path.read_text(encoding="utf-8")


business = load_section("business")
risks = load_section("risk_factors")
mdna = load_section("mdna")

for name, text in [("Business", business), ("Risk Factors", risks), ("MD&A", mdna)]:
    print(
        f"{name:<13} {len(text):>7,} chars  -- {shorten(text.replace(chr(10), ' '), width=120)}"
    )

Business       16,216 chars  -- # Source: SEC EDGAR 0000320193-25-000079 (10-K) # Company: Apple Inc. (CIK 320193, ticker AAPL) # Filed: [...]
Risk Factors   68,329 chars  -- # Source: SEC EDGAR 0000320193-25-000079 (10-K) # Company: Apple Inc. (CIK 320193, ticker AAPL) # Filed: [...]
MD&A           18,176 chars  -- # Source: SEC EDGAR 0000320193-25-000079 (10-K) # Company: Apple Inc. (CIK 320193, ticker AAPL) # Filed: [...]


### Helper: trim long sections

LLM context windows are large but not free. We truncate any single section to ~12,000 characters (~3,000 tokens) by keeping its first and last 6,000 characters, so each call is fast and cheap. For real production work you would chunk + summarize iteratively instead of truncating; we keep things simple here for teaching.

:::{caution} The model only sees an excerpt

Trimming hides most of a long section. The Risk Factors section is about 68,000 characters, so the model sees only about a sixth of it: the opening and the closing pages. Any risk discussed in the middle is invisible to the model. Keep this in mind when reading the risk classification in section 8.

:::


In [6]:
MAX_CHARS = 12_000


def trim(text: str, max_chars: int = MAX_CHARS) -> str:
    if len(text) <= max_chars:
        return text
    head = text[: max_chars // 2]
    tail = text[-max_chars // 2 :]
    return head + "\n\n[... truncated for brevity ...]\n\n" + tail


business_t = trim(business)
risks_t = trim(risks)
mdna_t = trim(mdna)

## 4. A small wrapper around `chat.completions.create`

Everything in this notebook is a chat-completion call. To avoid repetition we wrap the call in a helper that:

1. Always passes `model`, `messages`, and (optionally) a `response_format` for structured output.
2. Returns the assistant message _and_ the token-usage report so we can keep an eye on cost.
3. Adds each call's token usage to a running log, `USAGE_LOG`, so section 12 can estimate the cost of the whole notebook.


In [ ]:
# Every call appends its token usage here, so section 12 can price the whole notebook.
# Restart the kernel and run all cells for a clean total: re-running a cell logs its call again.
USAGE_LOG: list[dict] = []


def chat(
    user_prompt: str,
    *,
    system: str = "You are a careful financial analyst who answers concisely.",
    model: str = DEFAULT_MODEL,
    response_format: dict | None = None,
    temperature: float = 0.2,
) -> tuple[str, dict]:
    """Send one chat-completion request and return (text, usage_dict)."""
    kwargs: dict = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
    }
    if response_format is not None:
        kwargs["response_format"] = response_format

    resp = client.chat.completions.create(**kwargs)
    text = resp.choices[0].message.content or ""
    usage = {
        "input_tokens": resp.usage.prompt_tokens,
        "output_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
    }
    USAGE_LOG.append({"model": model, **usage})
    return text, usage

## 5. Free-form task #1: One-paragraph executive summary

We start with the simplest possible task: ask the model for a short, narrative summary of the **Business** section. There is no schema and no constraints, just plain prose.


In [8]:
prompt = (
    "Write a single, tight paragraph (no more than 120 words) that explains, in plain "
    "language, what business this company is in. Avoid jargon and avoid quoting the "
    "filing verbatim.\n\n"
    "=== BUSINESS SECTION ===\n"
    f"{business_t}"
)

summary, usage = chat(prompt)
print(summary)
print("\nUsage:", usage)

Apple makes and sells consumer technology products and the software and services that go with them. Its main products are iPhones, Macs, iPads, Apple Watches, AirPods, and other devices, along with items like Apple TV and HomePod. It also earns money from services such as the App Store, cloud storage, Apple Music, Apple TV+, AppleCare support, advertising, and payment tools like Apple Pay and Apple Card. The company sells to consumers, schools, businesses, and governments through its stores, website, sales teams, carriers, and other resellers.

Usage: {'input_tokens': 2310, 'output_tokens': 117, 'total_tokens': 2427}


## 6. Free-form task #2: Plain-English MD&A explainer

The _Management's Discussion and Analysis_ section is dense and full of accounting language. Ask the model to explain it as if to a smart business student who has never read a 10-K.


In [9]:
prompt = (
    "You are tutoring a college student who has never read a 10-K before. Read the "
    "following Management's Discussion and Analysis (MD&A) section and explain, in "
    "5–7 short bullet points, what management is saying about how the business "
    "performed and why. Use plain English; define any accounting terms you use.\n\n"
    "=== MD&A ===\n"
    f"{mdna_t}"
)

mdna_explainer, usage = chat(prompt)
print(mdna_explainer)
print("\nUsage:", usage)

- **Apple grew overall in 2025.** Total net sales were **$416.2 billion**, up **6%** from 2024. In plain English, Apple sold more stuff and services than the year before.

- **Services was the fastest-growing business.** Services revenue rose **14%**. “Services” means things like the App Store, subscriptions, cloud, and other recurring digital offerings. This suggests Apple is making more money from ongoing customer usage, not just hardware sales.

- **iPhone still drove a lot of the business.** iPhone sales increased **4%**, mainly because Apple sold more **Pro models**. That means higher-end phones were a key reason revenue went up.

- **Mac also had a strong year.** Mac sales rose **12%**, helped by higher demand. iPad sales were up **5%** too, while **Wearables, Home and Accessories** fell **4%**, so not every product line grew.

- **Results differed by region.** Sales rose in the **Americas, Europe, Japan, and Rest of Asia Pacific**, but **Greater China fell 4%**. Apple says China

## 7. Free-form task #3: Sentiment narrative on MD&A

A classic financial-NLP task: how _positive_ or _negative_ is management's tone? We ask for a short narrative judgment plus a 1–5 score embedded in the prose. We will ask for a tone score again, with strict structured output, in section 10, and compare the two in section 11.


In [10]:
prompt = (
    "Read the MD&A excerpt below. In 3–4 sentences, describe management's overall "
    "tone (optimistic, neutral, cautious, defensive, etc.) and end your answer with "
    "a single line of the form `Score: X/5` where 1 = very negative and 5 = very "
    "positive.\n\n"
    "=== MD&A ===\n"
    f"{mdna_t}"
)

sentiment_text, usage = chat(prompt, model=CHEAP_MODEL)
print(sentiment_text)
print("\nUsage:", usage)

Management’s tone is largely neutral-to-cautious: it highlights steady growth in net sales and specific drivers by segment and product category, but repeatedly emphasizes uncertainty and potential downside from macroeconomic factors. The discussion on tariffs and international trade measures is particularly cautious, noting material adverse impacts are possible and that the ultimate effects remain uncertain. Overall, management balances positive operating commentary with extensive risk disclosures around currency, consumer sentiment, supply chain inputs, taxes, and legal contingencies.  
Score: 3/5

Usage: {'input_tokens': 2826, 'output_tokens': 103, 'total_tokens': 2929}


## 8. Structured task #1: Risk-factor classification (JSON schema)

Now we shift to **structured output**. We give the model a JSON schema that forces the response to be a list of objects, each with a `theme`, a `severity` (low/medium/high), and a one-sentence `description`. Because the schema is enforced, we can `json.loads()` the response and use it directly.

The prompt asks the model to judge severity by how much space the filing gives each risk. Because of the trimming above, the model can only judge the space within the excerpt it sees, not within the full section.

We use the OpenAI [JSON Schema response format](https://platform.openai.com/docs/guides/structured-outputs):

```python
response_format = {
    "type": "json_schema",
    "json_schema": {"name": "...", "strict": True, "schema": {...}},
}
```

Most OpenAI-compatible providers (including OpenRouter routes that proxy to OpenAI/Mistral/Gemini-class models) honor the same shape. If your chosen model does not support structured outputs, you can fall back to `response_format={"type": "json_object"}` plus an explicit instruction in the prompt to return JSON.


In [11]:
risk_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "risk_classification",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "required": ["risks"],
            "properties": {
                "risks": {
                    "type": "array",
                    "minItems": 5,
                    "maxItems": 8,
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "required": ["theme", "severity", "description"],
                        "properties": {
                            "theme": {"type": "string"},
                            "severity": {
                                "type": "string",
                                "enum": ["low", "medium", "high"],
                            },
                            "description": {"type": "string"},
                        },
                    },
                }
            },
        },
    },
}

prompt = (
    "Read the Risk Factors section below and identify 5–8 distinct risk THEMES "
    "(e.g., 'Supply chain concentration', 'Foreign exchange exposure', "
    "'Regulatory / antitrust'). For each theme, assign a severity (low/medium/high) "
    "based on how much space and emphasis the filing gives it, and write a "
    "one-sentence plain-English description.\n\n"
    "=== RISK FACTORS ===\n"
    f"{risks_t}"
)

risk_json_text, usage = chat(prompt, response_format=risk_schema)
print(risk_json_text[:500], "...")
print("\nUsage:", usage)

{"risks":[{"theme":"Global macroeconomic conditions and consumer demand","severity":"high","description":"A slowdown, recession, inflation, higher rates, or weaker consumer confidence could reduce demand for Apple’s products and services and hurt results."},{"theme":"Geopolitical, trade, and supply-chain disruption","severity":"high","description":"Tariffs, export/import restrictions, conflict, terrorism, disasters, or other disruptions could raise costs, limit product availability, and force Ap ...

Usage: {'input_tokens': 2304, 'output_tokens': 364, 'total_tokens': 2668}


In [12]:
risk_data = json.loads(risk_json_text)
risk_data["risks"][:3]

[{'theme': 'Global macroeconomic conditions and consumer demand',
  'severity': 'high',
  'description': 'A slowdown, recession, inflation, higher rates, or weaker consumer confidence could reduce demand for Apple’s products and services and hurt results.'},
 {'theme': 'Geopolitical, trade, and supply-chain disruption',
  'severity': 'high',
  'description': 'Tariffs, export/import restrictions, conflict, terrorism, disasters, or other disruptions could raise costs, limit product availability, and force Apple to rework its global supply chain.'},
 {'theme': 'International manufacturing and supplier concentration',
  'severity': 'high',
  'description': 'Because much of Apple’s manufacturing and many suppliers are concentrated outside the U.S., problems in key countries or with key partners could quickly affect production and sales.'}]

Because the response is structured, we can drop it straight into a DataFrame and sort, filter, or chart it like any other tabular data.


In [13]:
import pandas as pd

risk_df = pd.DataFrame(risk_data["risks"])
risk_df = risk_df.sort_values(
    "severity",
    key=lambda s: s.map({"high": 0, "medium": 1, "low": 2}),
).reset_index(drop=True)
risk_df

,theme,severity,description
0,Global macroeconomic conditions and consumer d...,high,"A slowdown, recession, inflation, higher rates..."
1,"Geopolitical, trade, and supply-chain disruption",high,"Tariffs, export/import restrictions, conflict,..."
2,International manufacturing and supplier conce...,high,Because much of Apple’s manufacturing and many...
3,Foreign exchange exposure,medium,Movements in currency exchange rates can chang...
4,Credit and collectibility risk,medium,"Apple could lose money if customers, carriers,..."
5,Investment portfolio and liquidity risk,medium,"Changes in interest rates, credit quality, or ..."
6,Tax law and tax authority uncertainty,medium,"Changes in tax rules, tax rates, or audit outc..."
7,Stock price volatility and market expectations,low,Apple’s share price may swing sharply if inves...


## 9. Structured task #2: Named-entity extraction

We ask the model to pull three lists of entities out of the _Business_ section:

- **products** mentioned by name,
- **competitors** explicitly named, and
- **geographies** (countries / regions) the company operates in.

Each list element includes a short `evidence` snippet so the user can audit where the model saw the entity in the source text.


In [14]:
entity_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "entity_extraction",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "required": ["products", "competitors", "geographies"],
            "properties": {
                bucket: {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "required": ["name", "evidence"],
                        "properties": {
                            "name": {"type": "string"},
                            "evidence": {"type": "string"},
                        },
                    },
                }
                for bucket in ("products", "competitors", "geographies")
            },
        },
    },
}

prompt = (
    "From the Business section below, extract three lists of entities:\n"
    "- products       : specific product or service names the company sells\n"
    "- competitors    : companies explicitly identified as competitors\n"
    "- geographies    : countries, regions, or markets the company explicitly operates in\n\n"
    "For each entity, include a short `evidence` snippet (<= 20 words) quoting the "
    "phrase from the filing that supports the extraction. Only include entities that "
    "are explicitly mentioned in the text. Do NOT guess.\n\n"
    "=== BUSINESS SECTION ===\n"
    f"{business_t}"
)

entity_json_text, usage = chat(prompt, response_format=entity_schema)
entities = json.loads(entity_json_text)

print("Products    :", len(entities["products"]))
print("Competitors :", len(entities["competitors"]))
print("Geographies :", len(entities["geographies"]))
print("\nUsage:", usage)

Products    : 17
Competitors : 0
Geographies : 15

Usage: {'input_tokens': 2479, 'output_tokens': 673, 'total_tokens': 3152}


In [15]:
# Quick peek
for bucket in ("products", "competitors", "geographies"):
    print(f"\n--- {bucket.upper()} ---")
    for e in entities[bucket][:5]:
        print(f"  {e['name']!r:35}  evidence: {e['evidence']!r}")


--- PRODUCTS ---
  'iPhone'                             evidence: '“iPhone® is the Company’s line of smartphones”'
  'Mac'                                evidence: '“Mac® is the Company’s line of personal computers”'
  'iPad'                               evidence: '“iPad® is the Company’s line of multipurpose tablets”'
  'Apple Watch'                        evidence: '“includes Apple Watch® Series 11”'
  'AirPods'                            evidence: '“The Company’s line of wireless headphones includes AirPods®”'

--- COMPETITORS ---

--- GEOGRAPHIES ---
  'Americas'                           evidence: '“reportable segments consist of the Americas”'
  'Europe'                             evidence: '“reportable segments consist of the Americas, Europe”'
  'Greater China'                      evidence: '“reportable segments consist of the Americas, Europe, Greater China”'
  'Japan'                              evidence: '“reportable segments consist of the Americas, Europe, Greater Chi

## 10. Structured task #3: Final "executive briefing"

For the capstone task we combine everything into a single structured _executive briefing_: a short summary of the company, a tone score for management, the top 3 risks, and a confidence level. This is the kind of object an analytics team would dump into a database, a Slack message, or a dashboard.


In [16]:
briefing_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "executive_briefing",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "required": [
                "company_one_liner",
                "management_tone",
                "tone_score",
                "top_risks",
                "confidence",
            ],
            "properties": {
                "company_one_liner": {"type": "string"},
                "management_tone": {
                    "type": "string",
                    "enum": [
                        "very negative",
                        "negative",
                        "neutral",
                        "positive",
                        "very positive",
                    ],
                },
                "tone_score": {"type": "integer", "minimum": 1, "maximum": 5},
                "top_risks": {
                    "type": "array",
                    "minItems": 3,
                    "maxItems": 3,
                    "items": {"type": "string"},
                },
                "confidence": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                },
            },
        },
    },
}

# We give the model just the previously extracted artifacts, since there's no need to re-send the full filing.
prompt = (
    "Produce a one-paragraph executive briefing for a portfolio manager. Base your "
    "answer ONLY on the materials below.\n\n"
    "=== BUSINESS SUMMARY (from a previous LLM call) ===\n"
    f"{summary}\n\n"
    "=== MD&A EXPLAINER (from a previous LLM call) ===\n"
    f"{mdna_explainer}\n\n"
    "=== RISK CLASSIFICATION (JSON, from a previous LLM call) ===\n"
    f"{json.dumps(risk_data, indent=2)}"
)

briefing_text, usage = chat(prompt, response_format=briefing_schema)
briefing = json.loads(briefing_text)
briefing

{'company_one_liner': 'Apple is a global consumer technology and services company with 2025 net sales of $416.2 billion, led by iPhone, Services, Mac, and a large installed base that supports recurring revenue.',
 'management_tone': 'positive',
 'tone_score': 4,
 'top_risks': ['Global macroeconomic conditions and weaker consumer demand could pressure product and services sales.',
  'Geopolitical, trade, and supply-chain disruption, including tariffs, could raise costs and affect availability and margins.',
  'Foreign exchange movements and Greater China weakness could continue to weigh on reported results.'],
 'confidence': 'high'}

Because every field has a known type, downstream code can act on it without parsing strings. For example, a team could route every briefing with a low tone score to a human reviewer:

```python
if briefing["tone_score"] <= 2:
    send_for_human_review(briefing, source_text=mdna)
```

Notice what this rule does **not** use: the model's `confidence` field. That value is the model's own description of its answer, not a measurement, so it should not decide whether a result is trusted. Section 11 shows checks you can rely on instead.


## 11. Verify the model's outputs

An LLM can return fluent, confident text that is wrong: a figure that is not in the filing, a quote the filing never says, or an entity it made up. This is usually called **hallucination**. Structured output makes answers easier to check, but it does not make them true. Before any result goes into a database, a dashboard, or a report, check it.

We run three checks that need no extra API calls:

1. Does each parsed JSON response match the schema we asked for?
2. Does each evidence quote from the entity extraction actually appear in the filing text?
3. Do the dollar figures the model wrote match the audited income statement saved by the first notebook?

:::{note} No stored output

This section was added after the stored run, so the cells below show no output on this page. Run the notebook to see the results.

:::

### 11.1 Schema checks

With OpenAI models, `"strict": True` makes the API enforce the schema. Not every provider or model honours strict schemas, and the `json_object` fallback described in section 8 only promises valid JSON, not the right fields. A short validator catches missing keys, wrong types, values outside an `enum`, and lists of the wrong length. It covers only the schema keywords used in this notebook; for anything larger, use a library such as `jsonschema` or `pydantic`.


In [ ]:
def schema_problems(value, schema: dict, path: str = "$") -> list[str]:
    """Return the ways `value` breaks `schema` (an empty list means it passed).

    Covers only the JSON Schema keywords used in this notebook.
    """
    py_types = {"object": dict, "array": list, "string": str, "integer": int}
    expected = schema.get("type")
    if expected:
        ok = isinstance(value, py_types[expected])
        if expected == "integer" and isinstance(value, bool):
            ok = False  # in Python, True and False are also ints
        if not ok:
            return [f"{path}: expected {expected}, got {type(value).__name__}"]

    problems = []
    if "enum" in schema and value not in schema["enum"]:
        problems.append(f"{path}: {value!r} is not one of {schema['enum']}")
    if expected == "integer":
        low, high = schema.get("minimum"), schema.get("maximum")
        if (low is not None and value < low) or (high is not None and value > high):
            problems.append(f"{path}: {value} is outside [{low}, {high}]")
    if expected == "array":
        n, low, high = len(value), schema.get("minItems", 0), schema.get("maxItems")
        if n < low or (high is not None and n > high):
            problems.append(f"{path}: has {n} items, expected {low} to {high}")
        for i, item in enumerate(value):
            problems += schema_problems(item, schema.get("items", {}), f"{path}[{i}]")
    if expected == "object":
        properties = schema.get("properties", {})
        for key in schema.get("required", []):
            if key not in value:
                problems.append(f"{path}: missing required key {key!r}")
        if schema.get("additionalProperties") is False:
            extra = sorted(set(value) - set(properties))
            if extra:
                problems.append(f"{path}: unexpected keys {extra}")
        for key, sub_schema in properties.items():
            if key in value:
                problems += schema_problems(value[key], sub_schema, f"{path}.{key}")
    return problems


for name, parsed, response_format in [
    ("Risk classification", risk_data, risk_schema),
    ("Entity extraction", entities, entity_schema),
    ("Executive briefing", briefing, briefing_schema),
]:
    problems = schema_problems(parsed, response_format["json_schema"]["schema"])
    print(f"{name:<20}: {'passes the schema' if not problems else problems}")

### 11.2 Do the evidence quotes appear in the filing?

The entity-extraction prompt asked for a short quote supporting each entity. A quote is only evidence if it is really in the text. We normalize quote marks, spacing, and capitalization, then look for each quote in the Business excerpt the model was given.

A quote that is not found may be paraphrased, shortened with "...", or invented. Each one needs a human look before the entity is used.


In [ ]:
def normalize(text: str) -> str:
    """Lower-case, use straight quote marks, and collapse whitespace."""
    for curly, straight in [("’", "'"), ("‘", "'"), ("“", '"'), ("”", '"')]:
        text = text.replace(curly, straight)
    return re.sub(r"\s+", " ", text).strip().lower()


# Compare against the excerpt the model was given (business_t), not the full section
source = normalize(business_t)

rows = []
for bucket in ("products", "competitors", "geographies"):
    for entity in entities[bucket]:
        quote = normalize(entity["evidence"]).strip("\"' ")
        rows.append(
            {
                "bucket": bucket,
                "name": entity["name"],
                "evidence": entity["evidence"],
                "found_in_source": quote in source,
            }
        )

evidence_df = pd.DataFrame(rows)
print(
    f"Evidence quotes found word for word in the source: "
    f"{evidence_df['found_in_source'].sum()} of {len(evidence_df)}"
)

# Quotes that need a human look
evidence_df[~evidence_df["found_in_source"]]

### 11.3 Cross-check dollar figures against the XBRL income statement

The MD&A explainer and the briefing quote dollar amounts. The first notebook saved the company's income statement, parsed from the XBRL data filed with the 10-K, to `extracted/`. Below we pull every "\$X billion" figure from the model's text and look for an income-statement line with the same value, allowing for rounding.

A figure with no match is not automatically wrong: share buybacks and dividends, for example, appear in the cash flow statement, not the income statement. But every unmatched figure must be checked against the filing by hand before anyone relies on it.


In [ ]:
income_path = extracted_dir / f"{TICKER}_income_statement.csv"

if not income_path.exists():
    print(f"{income_path} not found. Re-run `read-edgar-filings.ipynb` to create it.")
else:
    income = pd.read_csv(income_path)
    # Period columns look like "2025-09-27 (FY)"; the largest date is the latest fiscal year
    latest_year = max(c for c in income.columns if c.endswith("(FY)"))
    xbrl = income[["label", latest_year]].dropna()
    xbrl["billions"] = xbrl[latest_year] / 1e9

    # Every "$X billion" figure the LLM wrote in the MD&A explainer and the briefing
    llm_text = (mdna_explainer + "\n" + briefing["company_one_liner"]).replace(",", "")
    figures = sorted(set(re.findall(r"\$(\d+(?:\.\d+)?) billion", llm_text)), key=float)

    checks = []
    for figure in figures:
        decimals = len(figure.split(".")[1]) if "." in figure else 0
        tolerance = 0.5 * 10**-decimals  # e.g. "$416.2 billion" may be rounded by up to 0.05
        hits = xbrl[(xbrl["billions"] - float(figure)).abs() <= tolerance]
        checks.append(
            {
                "llm_figure": f"${figure} billion",
                "matching_xbrl_lines": ", ".join(hits["label"].unique()) or "(no match)",
            }
        )

    print(f"Income statement column used: {latest_year}")
    display(pd.DataFrame(checks))

### 11.4 Two tone scores that disagree

Task 3 and Task 6 both judge management's tone, and in the run shown above they disagree. Task 3 called the MD&A "neutral-to-cautious" and scored it 3 out of 5. Task 6 called it "positive", scored it 4, and rated its own confidence "high".

Several things differ between the two calls:

- **Different inputs.** Task 3 read an excerpt of the MD&A itself. Task 6 read only earlier LLM outputs: a business summary, an explainer that leads with growth figures, and a list of risk themes. Summaries tend to drop management's hedging language, so a model reading them sees a more upbeat document.
- **Different models.** Task 3 used the cheap model; Task 6 used the default model.
- **No shared scale.** Neither prompt defines what a 3 or a 4 means, so each call applies its own idea of the scale.

Neither score is "the" answer. An LLM tone score is a judgement, not a measurement. To use one in practice, define the scale in the prompt with examples, score the original text rather than summaries of it, keep the model fixed, run the same prompt several times to see how much the score moves, and compare it with a simple benchmark such as a finance word list or a handful of human-labelled examples.

The `confidence: "high"` field does not settle the conflict. It came out of the same generation process as the score, and it was "high" even though another call disagreed. Gate decisions on checks you can run, like the three above, not on what the model says about itself.

:::{warning} Hallucination is a normal failure mode, not a rare bug

The checks above catch some problems, not all. A summary can be wrong without containing a single number or quote, and an entity list can silently leave things out. For anything that matters (an investment memo, a compliance alert), keep a person in the loop who reads the model's output next to the source text.

:::


## 12. Cost awareness

Token usage is reported on every response under `resp.usage`, and our `chat()` helper adds each call to `USAGE_LOG`. Below we total the log by model and price it, which estimates the cost of running this entire notebook end-to-end.

The prices are the per-token prices OpenRouter's model list showed for `openai/gpt-5.4-mini` and `openai/gpt-5.4-nano` on 2026-11-13, converted to USD per 1M tokens. **Always check your provider's own pricing page and update the table** before relying on the estimate: model prices change frequently, and providers can differ.


In [ ]:
# USD per 1M tokens, as listed for OpenAI's models in OpenRouter's model API
# (https://openrouter.ai/api/v1/models) on 2026-11-13.
# Prices change often: check your provider's own pricing page before relying on these.
PRICE_PER_1M = {
    "gpt-5.4-mini": {"input": 0.75, "output": 4.50},
    "gpt-5.4-nano": {"input": 0.20, "output": 1.25},
}


def estimate_cost(input_tokens: int, output_tokens: int, model: str) -> float:
    """Estimated cost in USD, or NaN if the model is not in PRICE_PER_1M."""
    price = PRICE_PER_1M.get(model.removeprefix("openai/"))  # OpenRouter names carry a prefix
    if price is None:
        return float("nan")
    return (input_tokens / 1_000_000) * price["input"] + (
        output_tokens / 1_000_000
    ) * price["output"]


usage_df = pd.DataFrame(USAGE_LOG)
cost_by_model = usage_df.groupby("model")[["input_tokens", "output_tokens"]].sum()
cost_by_model["calls"] = usage_df.groupby("model").size()
cost_by_model["est_cost_usd"] = [
    estimate_cost(row["input_tokens"], row["output_tokens"], model)
    for model, row in cost_by_model.iterrows()
]
display(cost_by_model)

print(f"Calls logged        : {len(usage_df)}")
print(f"Total estimated cost: ${cost_by_model['est_cost_usd'].sum():.4f}")
if cost_by_model["est_cost_usd"].isna().any():
    print("Add a price for each model shown as NaN to PRICE_PER_1M to include it in the total.")

:::{tip} Watch the ratio
Output tokens cost more than input tokens. For `gpt-5.4-mini` and `gpt-5.4-nano`, output costs about 6× as much as input (see the table above), and across the models on OpenRouter the ratio commonly falls between about 3× and 5×. If you find yourself paying mostly for _output_ (e.g., asking the model to repeat the filing back to you), redesign the prompt to ask for a _summary_ or a _structured extraction_ instead.
:::


## 13. Optional - Switching providers (OpenRouter & friends)

Because the `openai` SDK is just an HTTP client with a particular request shape, you can point it at any OpenAI-compatible provider by changing two values. With this notebook, you do that through the environment variables from section 2. No code changes are needed:

```bash
export OPENAI_API_KEY="sk-or-..."                    # your OpenRouter key
export OPENAI_BASE_URL="https://openrouter.ai/api/v1"
export LLM_MODEL="openai/gpt-5.4-mini"                # OpenRouter's name for the same model
export LLM_MODEL_CHEAP="openai/gpt-5.4-nano"
```

A running Jupyter server does not see environment variables you set later, so start Jupyter again from the terminal where you set them. Every helper above (`chat`, `risk_schema`, `entity_schema`, `briefing_schema`) continues to work unchanged. If your target model on OpenRouter does not support strict JSON schemas, downgrade `response_format` to `{"type": "json_object"}` and add an explicit _"Respond ONLY with valid JSON matching this shape: {…}"_ instruction in the user prompt. The checks in section 11 matter even more in that case.

:::{seealso} Browse OpenRouter models
The full list of available models, with live per-token prices, lives at <https://openrouter.ai/models>. Filter by _Tools_ and _JSON mode_ support if you plan to rely on structured outputs.
:::


## 14. Recap

In this notebook we:

- Connected the `openai` Python client to either OpenAI or any OpenAI-compatible provider by setting `api_key` and `base_url` from environment variables, with automatic retries for rate limits.
- Ran **three free-form** chat-completion tasks: business summary, MD&A explainer, and a sentiment narrative.
- Ran **three structured** tasks using JSON Schema response formats: risk classification, entity extraction, and a final executive briefing.
- **Checked** the structured outputs: schema validation, evidence quotes looked up in the source text, and dollar figures compared with the XBRL income statement. We also saw two tone scores for the same MD&A disagree, and why self-reported confidence cannot settle that.
- Logged token usage on every call and estimated the cost of the whole notebook from a per-million-token price table.

Combined with the EDGAR notebook, you now have a tiny end-to-end pipeline that turns a raw 10-K into machine-readable analytical artifacts, plus the checks needed before anyone relies on them. Financial research firms, audit teams, and investor-relations groups use the same pattern to triage large numbers of filings.
